# 合成CSVデータのスクリーニング可視化・件数確認ノート

このノートでは、以下を**探索用途**で確認します（学習は行いません）。

1. 指定ディレクトリ内の CTMC CSV を読み込み、スクリーニング前後の件数を確認する
2. `Q' (= Q_mle)` から抽出した `λ_i = Q'[i, i+1]` の分布を可視化し、スクリーニングの妥当性を点検する

- **入力**: `data_dir`（CSV格納ディレクトリ）
- **出力**: 保持/除外件数、除外理由集計、`λ` 可視化（N混在対応）

In [19]:
# ==== 最初にここを編集 ====
# data_dir: CSV ファイルがあるディレクトリ
# recursive: サブディレクトリまで探索するか

data_dir = "/mnt/ssd/datas/discrete_train_with_mle/"  # <- 最初にここを編集
recursive = True

# スクリーニング閾値
min_lambda = 1e-8
max_lambda = 1e6

# スクリーニング設定のON/OFF
check_nan_inf = True
require_structure = True
max_abs_diag_diff = 1e-8

# 表示設定
top_k_dropped_examples = 10
# 可視化向けに full parse する最大件数（kept/dropped それぞれ）
max_visualize_per_group = 10000


In [2]:
from __future__ import annotations

import sys
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import japanize_matplotlib

import numpy as np

# ノートブックの実行場所に依らず import しやすくする
repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ctmc_surrogate.data.dataset_csv_loader import parse_ctmc_csv
from ctmc_surrogate.data.dataset_screening import (
    ScreeningConfig,
    extract_lambdas_from_Q,
    screen_dir_fast,
)


In [3]:
# まずは samples を読まずに高速スクリーニング
cfg = ScreeningConfig(
    min_lambda=min_lambda,
    max_lambda=max_lambda,
    check_nan_inf=check_nan_inf,
    require_structure=require_structure,
    max_abs_diag_diff=max_abs_diag_diff,
)

fast = screen_dir_fast(data_dir, recursive=recursive, cfg=cfg)

print(f"総件数: {fast['total']}")
print(f"kept 件数: {len(fast['kept_paths'])}")
print(f"dropped 件数: {len(fast['dropped_paths'])}")

print("\n[dropped 理由別カウント]")
if fast['drop_counts']:
    for reason, cnt in sorted(fast['drop_counts'].items(), key=lambda x: x[1], reverse=True):
        print(f"- {reason}: {cnt}")
else:
    print("- dropped はありません")

print(f"\n[dropped 上位 {top_k_dropped_examples} 件]")
for i, item in enumerate(fast['report'][:top_k_dropped_examples], start=1):
    path = item.get("path", "(pathなし)")
    reason = item.get("reason", "(reasonなし)")
    idx = item.get("index")
    lam = item.get("lambda")
    detail = item.get("detail")
    base = f"{i:02d}. path={path}, reason={reason}"
    if idx is not None and lam is not None:
        base += f", index={idx}, lambda={lam:.6e}"
    if detail:
        base += f", detail={detail}"
    print(base)


総件数: 199936
kept 件数: 176521
dropped 件数: 23415

[dropped 理由別カウント]
- lambda_too_large: 12753
- nan_or_inf: 9418
- lambda_too_small: 1244

[dropped 上位 10 件]
01. path=/mnt/ssd/datas/discrete_train_with_mle/0_4163_4.csv, reason=lambda_too_large, index=0, lambda=2.746360e+15
02. path=/mnt/ssd/datas/discrete_train_with_mle/100001_3608_4.csv, reason=lambda_too_large, index=0, lambda=1.786465e+28
03. path=/mnt/ssd/datas/discrete_train_with_mle/100011_4798_4.csv, reason=nan_or_inf
04. path=/mnt/ssd/datas/discrete_train_with_mle/100021_2071_4.csv, reason=lambda_too_large, index=0, lambda=4.336674e+09
05. path=/mnt/ssd/datas/discrete_train_with_mle/100045_3496_4.csv, reason=nan_or_inf
06. path=/mnt/ssd/datas/discrete_train_with_mle/100064_4000_4.csv, reason=lambda_too_large, index=0, lambda=1.471759e+15
07. path=/mnt/ssd/datas/discrete_train_with_mle/100066_4658_4.csv, reason=nan_or_inf
08. path=/mnt/ssd/datas/discrete_train_with_mle/100071_2436_4.csv, reason=lambda_too_large, index=0, lambda=1.36